In [2]:
import pandas as pd
import joblib
import os
from multiprocessing import Pool
import re
from difflib import SequenceMatcher  # for longest common substring
from functools import partial
from operator import itemgetter
import Levenshtein  # levenstein/edit distance; docs here: https://rawgit.com/ztane/python-Levenshtein/master/docs/Levenshtein.html
import numpy as np
from sklearn.metrics import roc_auc_score


In [3]:
data = pd.read_csv('../../Potomac/AIT 620/LabWorks/PythonProject/datafiles/forbes_freebase_goldstandard_train.csv', names=['string1', 'string2', 'matched'])

In [4]:
data.head()

,string1,string2,matched
0,General Electric,General Electric,True
1,Wells Fargo,Wells Fargo,True
2,Bank of China,Industrial and Commercial Bank of China (Asia),True
3,PetroChina,PetroChina,True
4,Apple,Apple Inc.,True


In [5]:
len(data)

213

In [6]:
def clean_string(string):
    '''We will use this functions to remove special characters etc before 
    any string distance calculation.
    '''
    return ''.join(map(lambda x: x.lower() if str.isalnum(x) else ' ', string)).strip()

def levenstein_distance(s1_, s2_):
    s1, s2 = clean_string(s1_), clean_string(s2_)
    len_s1, len_s2 = len(s1), len(s2)
    return Levenshtein.distance(
        s1, s2
    ) / max([len_s1, len_s2])

def jaro_winkler_distance(s1_, s2_):
    s1, s2 = clean_string(s1_), clean_string(s2_)
    return Levenshtein.jaro_winkler(s1, s2)

def common_substring_distance(s1_, s2_):
    s1, s2 = clean_string(s1_), clean_string(s2_)
    len_s1, len_s2 = len(s1), len(s2)
    match = SequenceMatcher(
        None, s1, s2
    ).find_longest_match(0, len_s1, 0, len_s2)
    len_s1, len_s2 = len(s1), len(s2)
    norm = max([len_s1, len_s2])
    return 1 - min([1, match.size / norm])


In [7]:
dists = np.zeros(shape=(len(data), 3))
for algo_i, algo in enumerate(
    [levenstein_distance, jaro_winkler_distance, common_substring_distance]
):
    for i, string_pair in data.iterrows():
        dists[i, algo_i] = algo(string_pair['string1'], string_pair['string2'])
        
    print('AUC for {}: {}'.format(
        algo.__name__, 
        roc_auc_score(data['matched'].astype(float), 1 - dists[:, algo_i])
    ))

AUC for levenstein_distance: 0.9508904955034385
AUC for jaro_winkler_distance: 0.045979545053782406
AUC for common_substring_distance: 0.9560042320578381


In [8]:
from multiprocessing import Pool
from functools import partial
from operator import itemgetter

def fuzzy_string_search(
    s1,
    string_list,
    string_compare,
    threshold=lambda sim: sim > 0.7,
    use_multiprocessing=False,
    n_jobs=1
):
    string_match = partial(string_compare, s2_=s1)

    if use_multiprocessing:
        # Create pool INSIDE the function, not globally
        with Pool(n_jobs) as pool:
            comparisons = pool.map(string_match, string_list)
    else:
        # Pure Python loop (safe in Jupyter + Windows)
        comparisons = [string_match(s) for s in string_list]

    index, element = max(enumerate(comparisons), key=itemgetter(1))

    return string_list[index] if threshold(element) else None


In [10]:
company_list = [
        'Blackrock',
        'Credit Suisse',
        'Goldman Sachs',
        'Bank of America/Meryll Lynch',
        'Morgan Stanley',
        'LEK',
        'JP Morgan',
        'Nomura',
        'BNP Paribas',
        'WPP',
        'Rothschild',
        'Allianz',
    ]
fuzzy_string_search(
    'SAP',
    company_list,
    jaro_winkler_distance,
    threshold=lambda dist: dist < 0.7,
    use_multiprocessing=False
)

'WPP'

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

ngram_featurizer = CountVectorizer(
    min_df=1,
    analyzer='char',
    ngram_range=(1, 1), # this is the range of ngrams that are to be extracted!
    preprocessor=clean_string
)
company_space = ngram_featurizer.fit_transform(
    np.concatenate(
        [data['string1'], data['string2']],
        axis=0
    )
)

In [12]:
#print("COMPANY SPACE", company_space)
company_space

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4003 stored elements and shape (426, 41)>

In [13]:
# string1cv = ngram_featurizer.transform(data['string1'])
# string2cv = ngram_featurizer.transform(data['string2'])

# def norm(string1cv):
#     return string1cv / string1cv.sum(axis=1)

# similarities = 1 - np.sum(np.abs(norm(string1cv) - norm(string2cv)), axis=1) / 2
# roc_auc_score(data['matched'].astype(float), similarities)

################################

string1cv = ngram_featurizer.transform(data['string1'])
string2cv = ngram_featurizer.transform(data['string2'])

# Convert matrix objects to numpy arrays if they're sparse matrices
string1cv = string1cv.toarray() if hasattr(string1cv, 'toarray') else np.asarray(string1cv)
string2cv = string2cv.toarray() if hasattr(string2cv, 'toarray') else np.asarray(string2cv)

# Check shapes before proceeding
print(f"string1cv shape: {string1cv.shape}")
print(f"string2cv shape: {string2cv.shape}")

def norm(string_cv):
    # Check if array is empty
    if string_cv.size == 0:
        return string_cv
    
    # Ensure the array is at least 2D
    if string_cv.ndim == 1:
        string_cv = string_cv.reshape(1, -1)
    
    # Add small epsilon to avoid division by zero
    row_sums = string_cv.sum(axis=1, keepdims=True) + 1e-10
    return string_cv / row_sums

similarities = 1 - np.sum(np.abs(norm(string1cv) - norm(string2cv)), axis=1) / 2
roc_auc_score(data['matched'].astype(float), similarities)


string1cv shape: (213, 41)
string2cv shape: (213, 41)


0.9299506259918885

In [14]:
def editops_featurizer(s1_, s2_):
    '''Counts the replace, insert and delete operations between two strings
    and normalizes these by maximum string length.

    This featurization could be interesting to find out which operation is
    most useful.
    '''
    s1, s2 = clean_string(s1_), clean_string(s2_)
    len_s1, len_s2 = len(s1), len(s2)
    ops = Levenshtein.editops(
        s1, s2
    )
    index_dict = {'insert': 0, 'replace': 1, 'delete': 2}
    features = np.zeros((3))
    for op in edit_ops:
        features[index_dict[op[0]]] += 1
    features / max([len_s1, len_s2])  
    return features

def common_substring_featurizer(s1_, s2_):
    '''Here we extract 1. the normalized length of the common substring
    and whether the common substring matches 2. the beginning or
    3. the end of a word.
    '''
    s1, s2 = clean_string(s1_), clean_string(s2_)
    len_s1, len_s2 = len(s1), len(s2)
    longer_string = s1 if len_s1 > len_s2 else s2
    norm = max([len_s1, len_s2])    
    match = SequenceMatcher(None, s1, s2).find_longest_match(0, len_s1, 0, len_s2)
    substring = s1[match.a: match.a + match.size]
    
    m1 = re.search(
        '(?:^|\s|[a-z])' + substring,
        longer_string
    )
    m2 = re.search(
        substring + '(?:[a-z]|\s|$)',
        longer_string
    )
    return min([1, match.size / norm]), m1 is not None, m2 is not None

In [17]:
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Dense, Lambda, Input
import tensorflow as tf
from tensorflow.keras import backend as K



def create_string_featurization_model(feature_dimensionality, output_dim=50):
    '''
    Use for string featurization in combination with siamese models.
    Just a non-linear projection as a way of reducing the feature dimensionality
    in a meaningful way.

    Parameters:
        feature_dimensionality - number of features coming from the vectorizer
          or string featurization function
        output_dim - dimensions of the embedding/projection that we are trying
          to create
    '''
    preprocessing_model = Sequential()
    preprocessing_model.add(
        Dense(output_dim, activation='selu', input_dim=feature_dimensionality)
    )
    preprocessing_model.summary()
    return preprocessing_model

def create_siamese_model(preprocessing_models, #initial_bias =
                          input_shapes=(10,)):
    def euclidean_distance(vects):
        x, y = vects
        x = K.l2_normalize(x, axis=-1)
        y = K.l2_normalize(y, axis=-1)
        sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
        return K.sqrt(K.maximum(sum_square, K.epsilon()))
    
    if not isinstance(preprocessing_models, (list, tuple)):
        raise ValueError('preprocessing models needs to be a list or tuple of models')

    print('{} models to be trained against each other'.format(len(preprocessing_models)))
    if not isinstance(input_shapes, list):
        input_shapes = [input_shapes] * len(preprocessing_models)
    
    inputs = []
    intermediate_layers = []
    for preprocessing_model, input_shape in zip(preprocessing_models, input_shapes):
        inputs.append(Input(shape=input_shape))
        intermediate_layers.append(preprocessing_model(inputs[-1]))

    layer_diffs = []
    for i in range(len(intermediate_layers)-1):        
        layer_diffs.append(
            Lambda(euclidean_distance)([intermediate_layers[i], intermediate_layers[i+1]])
        )    
    siamese_model = Model(inputs=inputs, outputs=layer_diffs)
    siamese_model.summary()
    return siamese_model

def compile_model(model):
    model.compile(
        optimizer='rmsprop',
        loss='mse',
        metrics=[
            #'accuracy',
            #tf.keras.metrics.FalseNegatives(name='fn'), 
            #tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            #tf.keras.metrics.AUC(name='auc'),
        ]
    )

# use like this:
# feature_dims = len(ngram_featurizer.get_feature_names())
# string_featurization_model = create_string_featurization_model(feature_dims, output_dim=50)

# siamese_model = create_siamese_model(
#     preprocessing_models=[string_featurization_model, string_featurization_model],
#     input_shapes=[(feature_dims,), (feature_dims,)],
# )
# compile_model(siamese_model)

feature_dims = len(ngram_featurizer.get_feature_names_out())

string_featurization_model = create_string_featurization_model(
    feature_dims, 
    output_dim=50
)

siamese_model = create_siamese_model(
    preprocessing_models=[string_featurization_model, string_featurization_model],
    input_shapes=[(feature_dims,), (feature_dims,)],
)

compile_model(siamese_model)


C:\Users\yathi\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 50)                  │           2,100 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,100 (8.20 KB)

 Trainable params: 2,100 (8.20 KB)

...

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 41)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_2 (InputLayer)    │ (None, 41)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential (Sequential)       │ (None, 50)                │           2,100 │ input_layer_1[0][0],       │
│                               │                           │                 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lambda (Lambda)               │ (None, 1)                 │               0 │ sequential[0][0],          │
│                               │                           │                 │ sequential[1][0]           │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,100 (8.20 KB)

 Trainable params: 2,100 (8.20 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
import numpy as np
from sklearn.metrics import roc_auc_score

string1 = ngram_featurizer.transform(data['string1']).toarray()
string2 = ngram_featurizer.transform(data['string2']).toarray()

def norm(x):
    return x / x.sum(axis=1, keepdims=True)

similarities = 1 - np.sum(np.abs(norm(string1) - norm(string2)), axis=1) / 2

roc_auc_score(data['matched'].astype(float), similarities)


0.9298183741844471

In [20]:
siamese_model.fit( 
    [string1cv, string2cv],
    1 - data['matched'].astype(float),
    epochs=1000
)

Epoch 1/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0985 - recall: 0.9533
Epoch 2/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0903 - recall: 0.9533 
Epoch 3/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0863 - recall: 0.9533 
Epoch 4/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0837 - recall: 0.9439 
Epoch 5/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0812 - recall: 0.9533 
Epoch 6/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0791 - recall: 0.9533
Epoch 7/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0771 - recall: 0.9533
Epoch 8/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0752 - recall: 0.9533 
Epoch 9/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0732 - recall: 0.9533 
Epoch 10/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0714 - recall: 0.9533
Epoch 11/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0698 - recall: 0.9533 
Epoch 12/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0682 - recall: 0

In [21]:
from scipy.spatial.distance import euclidean

string_rep1 = string_featurization_model.predict(
    ngram_featurizer.transform(data['string1'])
)
string_rep2 = string_featurization_model.predict(
    ngram_featurizer.transform(data['string2'])
)
dists = np.zeros(shape=(len(data), 1))
for i, (v1, v2) in enumerate(zip(string_rep1, string_rep2)):
    dists[i] = euclidean(v1, v2)
    
roc_auc_score(data['matched'].astype(float), 1 - dists)



7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


0.9805589843061189

In [22]:
# some very basic plotting stuff
# maybe useful for training

import matplotlib.pyplot as plt
import matplotlib as mpl


def plot_metrics(history, metrics=['loss', 'auc', 'precision', 'recall']):
    for n, metric in enumerate(metrics):
        name = metric.replace("_"," ").capitalize()
        
    mpl.rcParams['figure.figsize'] = (12, 10)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    plt.subplot(2,2,n+1)
    plt.plot(history.epoch,  history.history[metric], color=colors[0], label='Train')
    plt.plot(history.epoch, history.history['val_'+metric],
             color=colors[0], linestyle="--", label='Val')
    plt.xlabel('Epoch')
    plt.ylabel(name)
    if metric == 'loss':
        plt.ylim([0, plt.ylim()[1]])
    elif metric == 'auc':
        plt.ylim([0.8,1])
    else:
        plt.ylim([0,1])

    plt.legend()


#plot_metrics(history, metrics=['recall'])

In [24]:
# from annoy import AnnoyIndex

# index = AnnoyIndex(company_space.shape[1], 'euclidean')
# for i, company in enumerate(company_list):
#     emb = ngram_featurizer.transform([company]).toarray().flatten()
#     index.add_item(i, emb)
    
# index.build(10)
# index.save('string_list.ann')

import numpy as np
from sklearn.neighbors import NearestNeighbors
import joblib   # for saving the index

# Build embedding matrix
embeddings = np.vstack([
    ngram_featurizer.transform([company]).toarray().flatten()
    for company in company_list
])

# Build nearest-neighbor index
nn_index = NearestNeighbors(
    n_neighbors=10,
    metric='euclidean',
    algorithm='auto'
)
nn_index.fit(embeddings)

# Save the index (replaces index.save('string_list.ann'))
joblib.dump({
    "nn_index": nn_index,
    "embeddings": embeddings,
    "company_list": company_list
}, "string_list_index.pkl")



['string_list_index.pkl']

In [25]:
def query_company(query_string, top_k=5):
    query_emb = ngram_featurizer.transform([query_string]).toarray()
    distances, indices = nn_index.kneighbors(query_emb, n_neighbors=top_k)
    return [(company_list[i], distances[0][j]) for j, i in enumerate(indices[0])]


In [26]:
company_list

['Blackrock',
 'Credit Suisse',
 'Goldman Sachs',
 'Bank of America/Meryll Lynch',
 'Morgan Stanley',
 'LEK',
 'JP Morgan',
 'Nomura',
 'BNP Paribas',
 'WPP',
 'Rothschild',
 'Allianz']

In [ ]:
v = ngram_featurizer.transform(['Allianz AG']).toarray().flatten()
#ngram_featurizer.transform(['HSBC']).toarray()
neighbours, distances = index.get_nns_by_vector(v, 3, search_k=-1, include_distances=True)

In [30]:
query_emb = ngram_featurizer.transform(['SAP']).toarray()
distances, indices = nn_index.kneighbors(query_emb, n_neighbors=5)

# Best match
best_match = company_list[indices[0][0]]

# All matches
all_matches = [company_list[i] for i in indices[0]]

print("Best:", best_match)
print("Top 5:", all_matches)


Best: WPP
Top 5: ['WPP', 'LEK', 'Nomura', 'JP Morgan', 'BNP Paribas']


In [31]:
results = [
    (company_list[i], distances[0][j])
    for j, i in enumerate(indices[0])
]


In [34]:
results

[('WPP', np.float64(2.0)),
 ('LEK', np.float64(2.449489742783178)),
 ('Nomura', np.float64(2.6457513110645907)),
 ('JP Morgan', np.float64(2.8284271247461903)),
 ('BNP Paribas', np.float64(3.1622776601683795))]